# Rebuilding a diffusion model: from noise to images

**Independent PyTorch walkthrough following NVIDIA DLI course experiments.** This notebook does not reproduce proprietary lesson text and does not claim to reproduce the missing historical weights. It illustrates the code in `src/diffusion_portfolio`. Run from the **repository root**.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))
import torch
import matplotlib.pyplot as plt
from diffusion_portfolio.ddpm import Diffusion
from diffusion_portfolio.model import ConditionalUNet
from diffusion_portfolio.data import get_loader
from diffusion_portfolio.media import grid_image, export_frames
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.is_available())


## 1 · Load and normalize training images

Fashion-MNIST contains grayscale clothing images. We use 32×32 images so the U-Net can downsample twice; image tensors are normalized to [-1, 1]. The first execution may download the dataset.


In [ ]:
loader, dataset = get_loader('fashion', batch_size=16)
images, labels = next(iter(loader))
display(grid_image(images, columns=8, tile=6))


## 2 · Forward Gaussian corruption

A noise schedule controls variance. We sample directly from `q(x_t|x_0)` using `alpha_bar_t`, avoiding explicit iteration for training. Keep the noise realization fixed in these illustrations to show a gradual progression.


In [ ]:
d = Diffusion(steps=150)
epsilon = torch.randn_like(images)
forward_frames = []
for step in [0, 10, 25, 50, 75, 100, 125, 149]:
    t = torch.full((len(images),), step, dtype=torch.long)
    noisy, _ = d.forward_noise(images, t, epsilon)
    forward_frames.append(noisy)
    print(f't={step:03d}, alpha_bar={d.alpha_bars[step].item():.4f}')
display(grid_image(forward_frames[-1], columns=8, tile=6))


## 3 · Conditional U-Net and timestep embeddings

The U-Net predicts the added noise, not the final image. A class label is encoded as an embedding. Class=-1 represents missing context for classifier-free guidance.


In [ ]:
model = ConditionalUNet(channels=1, classes=10, base=16)
t = torch.randint(0, d.steps, (len(images),))
predicted_noise = model(images, t, labels)
print('Input:', images.shape, 'Noise estimate:', predicted_noise.shape)


## 4 · Training objective (single demonstration step)

Minimize mean-squared error between known sampled Gaussian noise and predicted noise. This step verifies backpropagation but is **not** a trained generator. For meaningful outputs train using `scripts/train.py`.


In [ ]:
optim = torch.optim.AdamW(model.parameters(), lr=2e-4)
model.train(); optim.zero_grad()
loss = d.loss(model, images, t, labels)
loss.backward(); optim.step()
print('One-step loss:', float(loss.item()))


## 5 · Sampling and generating GIF/MP4

**Important:** a randomly initialized or one-step-trained model produces noise, not credible clothing images. To see a realistic experiment run the full training CLI, load your checkpoint and save the complete reverse trajectory.


In [ ]:
# Run in your terminal after training:
# python scripts/train.py --dataset fashion --epochs 20
# python scripts/sample.py --checkpoint checkpoints/fashion.pt --samples 70 --output demo/generated_fashion
# python scripts/forward.py --dataset fashion --output demo/forward_fashion


## 6 · Classifier-free guidance

During training, drop each class with probability 0.1. At sampling time evaluate conditional and unconditional branches and combine:

`epsilon_guided = epsilon_uncond + guidance * (epsilon_cond - epsilon_uncond)`

A guidance strength of 1 matches conditional output, while 0 yields unconditional sampling. Sweeping guidance values can reveal the trade-off in class adherence and sample diversity.


## 7 · Flowers and CLIP extension

Extract the flower dataset locally under `data/cropped_flowers/` and train a 3-channel U-Net. The separate `scripts/clip_similarity.py` demonstrates text-image comparison using CLIP; it does **not** supply text conditioning to the diffusion model. See `docs/WORKFLOW.md` for commands and interpretation.


## 8 · Interpretation and provenance

The preserved files `demo/fashion.mp4` and `demo/flowers.mp4` come from historical course experimentation; the original model checkpoints have not been recovered. Newly trained samples should be named `demo/generated_*.mp4` and documented with measured loss curves, epochs, hardware and seed.
